# IOAI — 2025 Summer National Emoji Epics (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2025/nyari-orszagos/feladatok/adatok/emoji-epics >/dev/null && git checkout -q
import pandas as pd
df=pd.read_csv('haio/2025/nyari-orszagos/feladatok/adatok/emoji-epics/emoji_dataset.csv').dropna(subset=['emojis'])
df=df[df['emojis'].astype(str).str.strip()!=''].sample(frac=1,random_state=42).reset_index(drop=True)
v=df.index%5==0; tr=df[~v]; te=df[v].reset_index(drop=True)
tr[['title','emojis']].to_csv('train.csv',index=False)
te.assign(id=range(len(te)))[['id','title']].to_csv('test.csv',index=False)
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Emoji Epics (Epikus Emojik)

> **HAIO 2025 — Summer National Finals (NLP, generation)**

Translate a **movie title** into an **emoji sequence**. Score = **BLEU** (emoji tokens) vs reference. `train.csv` (title,emojis) is labelled; `test.csv` (id,title) is graded via **Submit** (`submission.csv`=`id,emojis`).

Baseline: TF-IDF title retrieval — copy the emojis of the most similar training title. (The original task fine-tunes an LLM; that's the improvement direction.)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
train=pd.read_csv('train.csv'); test=pd.read_csv('test.csv')
train['title']=train['title'].astype(str); test['title']=test['title'].astype(str)
vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,4))
Xtr=vec.fit_transform(train['title']); Xte=vec.transform(test['title'])
nn=NearestNeighbors(n_neighbors=1,metric='cosine').fit(Xtr)
_,idx=nn.kneighbors(Xte)
pred=train['emojis'].to_numpy()[idx[:,0]]
pd.DataFrame({'id':test['id'],'emojis':pred}).to_csv('submission.csv',index=False)
print('wrote',len(test),'| sample:',pred[0])

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)